# Stage 6 - physical threat-sizing (integer + per-ID envelope)

How much PGD degradation survives real-frame constraints. Round each adversarial frame to legal integers (per-feature clip, so the ID is not crushed), then reject any frame whose bytes fall outside the observed per-ID envelope of benign traffic.

In [1]:
import sys
from pathlib import Path
try:
    import adversec
except ModuleNotFoundError:
    sys.path.insert(0, str(Path.cwd().parent)); import adversec
import numpy as np, pandas as pd
from adversec import config
from adversec.contract import FEATURES, LABEL_COLUMN, ID_COLUMN, DATA_COLUMNS
pd.set_option('display.width', 140)

# The two datasets are treated identically: every step below runs the SAME
# code for both. The only dataset-specific code in the project is each
# dataset's loader (adversec/datasets/ciciov.py, road.py).
DATASETS = ['ciciov2024', 'road']

## Setup

In [2]:
import torch, joblib
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from adversec.models import CNN1D, train_cnn
from adversec.experiments import attack as atk, realism
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device:', DEVICE)
def mf1(y, p): return f1_score(y, p, average='macro', zero_division=0)
id_idx = FEATURES.index(ID_COLUMN); data_idx = [FEATURES.index(c) for c in DATA_COLUMNS]

device: cuda


## Sweep: raw vs integer-rounded F1, and % rejected by the per-ID envelope
Rounding that does NOT recover F1 means the threat is real on the discrete grid; a high rejection rate means a cheap validity check blocks most naive attacks.

Then, per dataset: an **adaptive, envelope-aware attacker** — one who already knows about
the per-ID envelope and deliberately stays inside it (ID frozen to a real observed value,
payload bytes clipped into that ID's own observed range). This tests the limitation the
threat-sizing note already names ("an adaptive attacker aware of the validator could
box-constrain perturbations to the per-ID envelope") instead of just asserting it.

In [3]:
threat_rows = {}
adaptive_rows = {}
models_for_bb = {}
for name in DATASETS:
    a = np.load(config.PROCESSED_DIR / f'{name}_stage2_arrays.npz')
    Xtr, ytr, Xte, yte = a['X_train'], a['y_train'], a['X_test'].astype(np.float32), a['y_test']
    classes = list(joblib.load(config.PROCESSED_DIR / f'{name}_label_encoder.joblib').classes_)
    scaler = joblib.load(config.PROCESSED_DIR / f'{name}_feature_scaler.joblib')
    cfg = config.load_dataset_config(name)
    cw = None
    if cfg.get('cnn_class_weights'):
        w = compute_class_weight('balanced', classes=np.unique(ytr), y=ytr)
        cw = torch.tensor(w, dtype=torch.float32, device=DEVICE)
    cnn = train_cnn(CNN1D(n_features=Xtr.shape[1], n_classes=len(classes)), Xtr, ytr, n_epochs=50, device=DEVICE, class_weights=cw)
    clf = atk.wrap_cnn_for_art(cnn, n_features=Xtr.shape[1], n_classes=len(classes), device=DEVICE)
    # Envelope from TRAIN benign only, so the false-positive check below (on held-out TEST
    # benign) and the adversarial-rejection rate are both measured on data the envelope
    # never saw -- building it from train+test would make the FP rate trivially optimistic.
    train_dup = pd.read_csv(config.PROCESSED_DIR / f'{name}_train_dup.csv')
    test_df   = pd.read_csv(config.PROCESSED_DIR / f'{name}_test.csv')
    benign = cfg['benign_label']
    train_benign = train_dup[train_dup[LABEL_COLUMN] == benign]
    test_benign = test_df[test_df[LABEL_COLUMN] == benign]
    ranges = realism.learn_observed_ranges(train_benign, ID_COLUMN, DATA_COLUMNS)

    benign_test_int = test_benign[FEATURES].values.astype(np.float64)
    benign_mask = realism.observed_range_mask(benign_test_int, ranges, id_idx, data_idx)
    benign_fp_rate = 100 * (~benign_mask).sum() / len(benign_mask) if len(benign_mask) else float('nan')

    print(f'\n=== {name}  (benign envelope: {len(ranges)} IDs, learned from TRAIN benign only) ===')
    print(f'    held-out TEST benign false-positive rate: {benign_fp_rate:.1f}%  '
          f'({int((~benign_mask).sum())}/{len(benign_mask)} legitimate frames wrongly rejected)')
    print(f"{'eps':>6}{'raw_F1':>9}{'round_F1':>10}{'%rejected':>11}{'survivors':>11}")
    rows = []
    for eps in config.FGSM_EPSILONS:
        Xadv = atk.generate_pgd(clf, Xte, eps)
        f1_raw = mf1(yte, clf.predict(Xadv).argmax(1))
        Xround, Xint = realism.round_to_integer_frames(Xadv, scaler)
        f1_round = mf1(yte, clf.predict(Xround).argmax(1))
        mask = realism.observed_range_mask(Xint, ranges, id_idx, data_idx)
        pct_rejected = 100 * (~mask).sum() / len(mask)
        rows.append({
            'eps': eps,
            'f1_raw_adversarial': round(float(f1_raw), 3),
            'f1_rounded_integer': round(float(f1_round), 3),
            'pct_rejected_by_envelope': round(float(pct_rejected), 1),
            'n_survivors': int(mask.sum()),
        })
        print(f'{eps:>6.2f}{f1_raw:>9.3f}{f1_round:>10.3f}{pct_rejected:>10.1f}%{int(mask.sum()):>11}')
    threat_rows[name] = {
        'benign_reference_frames': int(len(train_benign)),
        'benign_ids': len(ranges),
        'test_frames': int(len(yte)),
        'benign_false_positive_rate_pct': round(float(benign_fp_rate), 2),
        'benign_false_positive_frames_tested': int(len(benign_mask)),
        'by_epsilon': rows,
    }

    # Adaptive, envelope-aware attacker: freeze the ID (perturb payload bytes only, so
    # the frame keeps a real observed ID) and clip perturbed bytes into that ID's own
    # observed range -- tests the "adaptive attacker" limitation named above but never
    # actually measured. Reuses the SAME trained clf/ranges/scaler, no retraining.
    id_frozen_mask = np.zeros(len(FEATURES), dtype=np.float32)
    id_frozen_mask[data_idx] = 1.0
    print(f'\n    adaptive envelope-aware attack (ID frozen to a real value, bytes clipped to that ID\'s observed range):')
    print(f"{'eps':>6}{'F1':>9}{'%rejected':>11}{'survivors':>11}")
    adaptive_out = []
    for eps in config.FGSM_EPSILONS:
        Xadv_id = atk.generate_pgd(clf, Xte, eps, mask=id_frozen_mask)
        _, Xint_id = realism.round_to_integer_frames(Xadv_id, scaler)
        Xint_clipped = realism.clip_to_id_envelope(Xint_id, ranges, id_idx, data_idx)
        Xclipped_scaled = scaler.transform(Xint_clipped).astype(np.float32)
        f1_adaptive = mf1(yte, clf.predict(Xclipped_scaled).argmax(1))
        mask_adaptive = realism.observed_range_mask(Xint_clipped, ranges, id_idx, data_idx)
        pct_rejected_adaptive = 100 * (~mask_adaptive).sum() / len(mask_adaptive)
        adaptive_out.append({
            'eps': eps,
            'f1_adaptive_attack': round(float(f1_adaptive), 3),
            'pct_rejected_by_envelope': round(float(pct_rejected_adaptive), 1),
            'n_survivors': int(mask_adaptive.sum()),
        })
        print(f'{eps:>6.2f}{f1_adaptive:>9.3f}{pct_rejected_adaptive:>10.1f}%{int(mask_adaptive.sum()):>11}')
    adaptive_rows[name] = {
        'description': (
            "ID frozen to its real (observed) value; payload bytes perturbed by PGD then "
            "clipped into that ID's own observed benign range before evaluation."
        ),
        'by_epsilon': adaptive_out,
    }

    # Kept for the black-box (HopSkipJump) section below -- avoids retraining.
    models_for_bb[name] = dict(clf=clf, ranges=ranges, scaler=scaler, Xte=Xte, yte=yte)

    epoch   1/50     loss 1.6049
    epoch   5/50     loss 0.1692
    epoch  10/50     loss 0.0133
    epoch  15/50     loss 0.0041
    epoch  20/50     loss 0.0027
    epoch  25/50     loss 0.0019
    epoch  30/50     loss 0.0015
    epoch  35/50     loss 0.0012
    epoch  40/50     loss 0.0011
    epoch  45/50     loss 0.0010
    epoch  50/50     loss 0.0010

=== ciciov2024  (benign envelope: 58 IDs, learned from TRAIN benign only) ===
    held-out TEST benign false-positive rate: 3.2%  (23/709 legitimate frames wrongly rejected)
   eps   raw_F1  round_F1  %rejected  survivors


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

  0.01    0.661     0.661     100.0%          0


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

  0.05    0.494     0.492     100.0%          0


...

  0.10    0.316       1.4%       7859


PGD - Batches:   0%|          | 0/250 [00:00<?, ?it/s]

  0.20    0.134       1.4%       7859


PGD - Batches:   0%|          | 0/250 [00:00<?, ?it/s]

  0.30    0.132       1.4%       7859


## Black-box attack (HopSkipJump) — does the envelope survive a non-gradient attacker?

Every attack so far (FGSM, PGD, and the adaptive variant above) uses the model's gradients
directly — a strong assumption about attacker capability. HopSkipJump is decision-based: it
only queries the model for its predicted label, never touches gradients. This tests whether
the robustness/envelope findings generalise to a genuinely black-box threat model, not just
a white-box one.

**Scope, stated up front**: HopSkipJump is a minimum-perturbation search, and at ART's
default budget (`max_iter=50, max_eval=10000` → ~500k queries per sample) it is far too
expensive to run at any real sample count. We use a much smaller budget
(`max_iter=15, max_eval=300, init_eval=30, init_size=30`, ART's defaults are roughly
30x higher) on a stratified subsample — up to 20 test signatures per class per dataset.
This trades attack strength for tractability: treat the resulting F1 as a **lower bound**
on what a well-resourced black-box attacker could achieve, not the attack's ceiling. Reuses
the classifier/envelope/scaler already built above — no retraining.

In [4]:
HSJ_MAX_PER_CLASS = 20
HSJ_KWARGS = dict(max_iter=15, max_eval=300, init_eval=30, init_size=30)

blackbox_rows = {}
rng = np.random.RandomState(config.RANDOM_SEED)
for name in DATASETS:
    m = models_for_bb[name]
    clf, ranges, scaler, Xte, yte = m['clf'], m['ranges'], m['scaler'], m['Xte'], m['yte']

    idx_by_class = {c: np.where(yte == c)[0] for c in np.unique(yte)}
    sub_idx = np.concatenate([
        rng.choice(idxs, size=min(HSJ_MAX_PER_CLASS, len(idxs)), replace=False)
        for idxs in idx_by_class.values()
    ])
    X_sub, y_sub = Xte[sub_idx], yte[sub_idx]

    clean_f1 = mf1(y_sub, clf.predict(X_sub).argmax(1))

    X_hsj = atk.generate_hopskipjump(clf, X_sub, **HSJ_KWARGS)
    hsj_f1 = mf1(y_sub, clf.predict(X_hsj).argmax(1))
    l2_dist = np.linalg.norm((X_hsj - X_sub).reshape(len(X_sub), -1), axis=1)

    _, X_hsj_int = realism.round_to_integer_frames(X_hsj, scaler)
    hsj_mask = realism.observed_range_mask(X_hsj_int, ranges, id_idx, data_idx)
    pct_rejected_hsj = 100 * (~hsj_mask).sum() / len(hsj_mask)

    # Same PGD comparison on the IDENTICAL subsample, at the "meaningful" eps=0.10.
    X_pgd = atk.generate_pgd(clf, X_sub, epsilon=0.10)
    pgd_f1 = mf1(y_sub, clf.predict(X_pgd).argmax(1))

    print(f'\n=== {name}: HopSkipJump on {len(X_sub)} stratified test signatures ===')
    print(f'    clean F1              = {clean_f1:.3f}')
    print(f'    HopSkipJump F1        = {hsj_f1:.3f}   (mean L2 perturbation = {l2_dist.mean():.4f}, '
          f'envelope rejects {pct_rejected_hsj:.1f}%)')
    print(f'    PGD @ eps=0.10 F1     = {pgd_f1:.3f}   (same subsample, for comparison)')

    blackbox_rows[name] = {
        'n_samples': int(len(X_sub)),
        'max_per_class': HSJ_MAX_PER_CLASS,
        'hopskipjump_budget': HSJ_KWARGS,
        'clean_f1': round(float(clean_f1), 3),
        'hopskipjump_f1': round(float(hsj_f1), 3),
        'hopskipjump_mean_l2_perturbation': round(float(l2_dist.mean()), 4),
        'hopskipjump_pct_rejected_by_envelope': round(float(pct_rejected_hsj), 1),
        'pgd_eps010_f1_same_subsample': round(float(pgd_f1), 3),
    }

PGD - Batches:   0%|          | 0/1 [00:00<?, ?it/s]


=== ciciov2024: HopSkipJump on 29 stratified test signatures ===
    clean F1              = 0.755
    HopSkipJump F1        = 0.067   (mean L2 perturbation = 0.2709, envelope rejects 86.2%)
    PGD @ eps=0.10 F1     = 0.184   (same subsample, for comparison)


PGD - Batches:   0%|          | 0/4 [00:00<?, ?it/s]


=== road: HopSkipJump on 100 stratified test signatures ===
    clean F1              = 1.000
    HopSkipJump F1        = 0.067   (mean L2 perturbation = 0.1112, envelope rejects 81.0%)
    PGD @ eps=0.10 F1     = 0.226   (same subsample, for comparison)


## Save threat-sizing results (merges into `<name>_adversarial_results.json`)
Adds/updates the `threat_sizing` section alongside whatever notebook 04 already saved (per-class PGD CV, distance-to-benign) — run in either order, nothing is clobbered.

In [5]:
import json
config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for name in DATASETS:
    path = config.RESULTS_DIR / f'{name}_adversarial_results.json'
    existing = json.loads(path.read_text()) if path.exists() else {'dataset': name}
    existing['threat_sizing'] = {
        **threat_rows[name],
        'note': 'per-feature integer clip (arbitration ID not crushed to a byte range); envelope learned from TRAIN benign only',
    }
    existing['adaptive_envelope_aware_attack'] = adaptive_rows[name]
    existing['blackbox_hopskipjump_attack'] = blackbox_rows[name]
    path.write_text(json.dumps(existing, indent=2))
    print('saved ->', path)

saved -> /home/koala/lab/adversec/results/ciciov2024_adversarial_results.json
saved -> /home/koala/lab/adversec/results/road_adversarial_results.json
